# $B^+\to K^+\pi^+\pi^-$ with efficiency and background

The toy and the likelihood share the same efficiency/background objects, so there is no manual pool weighting or background normalization in the analysis code.


In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    FitSession, ToyBackground, enable_x64, generate_toy,
    plot_dalitz, plot_square_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


In [ ]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
rho_x = Parameter.coefficient("rho.x", 0.55, bounds=(-2,2), owner="rho", step=0.02)
rho_y = Parameter.coefficient("rho.y", 0.10, bounds=(-2,2), owner="rho", step=0.02)
model = DecayModel(
    channel,
    [
        Resonance("Kstar892",(0,2),RealImag(1.0,0.0),
                  mass=0.8958,width=0.0474,spin=1),
        Resonance("rho",(1,2),RealImag(rho_x,rho_y),
                  mass=0.7753,width=0.1491,spin=1),
        NonResonant(RealImag(-0.25,0.10)),
    ],
    normalization_method="square-dalitz",
    normalization_resolution=220,
    normalization_pair=(0,2),
)
truth={"rho.x":0.55,"rho.y":0.10}

efficiency = FunctionalEfficiency(
    lambda d: 0.50 + 0.35*jnp.clip(d["s13"]/20.0,0,1)
)
background = FunctionalBackground(
    lambda d: 0.35 + 0.8*jnp.clip(d["s23"]/25.0,0,1)
)
f_sig = Parameter("signal_fraction",0.78,bounds=(0.05,0.99),step=0.01)


In [ ]:
data = generate_toy(
    model, 35_000, parameters=truth, efficiency=efficiency,
    signal_fraction=0.82,
    backgrounds=(ToyBackground("combinatorial", background),),
    seed=404, pool_size=250_000,
)
plot_dalitz(data, x="s13", y="s23", title="Selected B+ pseudo-data")
plt.show()


In [ ]:
from dalitzplotfitter import BackgroundSpec

session = FitSession(
    model, data, efficiency=efficiency, signal_fraction=f_sig,
    backgrounds=(BackgroundSpec("combinatorial", background),),
)
result = session.fit({"rho.x":0.30,"rho.y":-0.10,"signal_fraction":0.70},
                     simplex=True,ncall=40_000)
session.report(result, acceptance_weighted_fractions=True)
session.plot_projection(result,"s13")
plt.show()
